In [1]:
!python -V

Python 3.14.3


In [2]:
import pickle

In [3]:
import pandas as pd

In [4]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import root_mean_squared_error

In [5]:
import mlflow

mlflow.set_experiment("nyc-taxi-experiment")

<Experiment: artifact_location='mlflow-artifacts:/3', creation_time=1777208039466, experiment_id='3', last_update_time=1777208039466, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, trace_location=None, workspace='default'>

In [6]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df["duration"] = df["lpep_dropoff_datetime"] - df["lpep_pickup_datetime"]
    df["duration"] = df["duration"].dt.total_seconds() / 60

    df = df[(df["duration"] >= 1) & (df["duration"] <= 60)]

    categorical = ["PULocationID", "DOLocationID"]
    df[categorical] = df[categorical].astype(str)

    df["PU_DO"] = df["PULocationID"] + "_" + df["DOLocationID"]

    return df

In [7]:
df_train = read_dataframe(
    "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet"
)
df_val = read_dataframe(
    "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet"
)

In [8]:
categorical = ['PU_DO']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [9]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [10]:
import xgboost as xgb

In [11]:
from pathlib import Path

In [12]:
models_folder = Path('models')
models_folder.mkdir(exist_ok=True)

In [13]:
mlflow.xgboost.autolog(disable=True)

with mlflow.start_run():

    train = xgb.DMatrix(X_train, label=y_train)
    valid = xgb.DMatrix(X_val, label=y_val)

    best_params = {
        "max_depth": 44,
        "learning_rate": 0.1719747846289316,
        "reg_alpha": 4.896614873911946e-05,
        "reg_lambda": 0.019446810721081013,
        "min_child_weight": 1.785047439733004,
        "objective": "reg:squarederror",
        "random_state": 42
    }

    mlflow.log_params(best_params)

    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=1000,
        evals=[(valid, 'validation')],
        early_stopping_rounds=50
    )

    y_pred = booster.predict(
        data=valid,
        iteration_range=(0, booster.best_iteration + 1)
    )
    error = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("error", error)

    with open("models/preprocessor.pkl", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("models/preprocessor.pkl", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, name="models_mlflow")


[0]	validation-rmse:10.84339
[1]	validation-rmse:9.78065
[2]	validation-rmse:8.96328
[3]	validation-rmse:8.34365
[4]	validation-rmse:7.87984
[5]	validation-rmse:7.53790
[6]	validation-rmse:7.27768
[7]	validation-rmse:7.08854
[8]	validation-rmse:6.94774
[9]	validation-rmse:6.84248
[10]	validation-rmse:6.76135
[11]	validation-rmse:6.70150
[12]	validation-rmse:6.65752
[13]	validation-rmse:6.61963
[14]	validation-rmse:6.59121
[15]	validation-rmse:6.56832
[16]	validation-rmse:6.54906
[17]	validation-rmse:6.53404
[18]	validation-rmse:6.52137
[19]	validation-rmse:6.51207
[20]	validation-rmse:6.50120
[21]	validation-rmse:6.49439
[22]	validation-rmse:6.48815
[23]	validation-rmse:6.48317
[24]	validation-rmse:6.47965
[25]	validation-rmse:6.47684
[26]	validation-rmse:6.47435
[27]	validation-rmse:6.47275
[28]	validation-rmse:6.47164
[29]	validation-rmse:6.46959
[30]	validation-rmse:6.46840
[31]	validation-rmse:6.46695
[32]	validation-rmse:6.46507
[33]	validation-rmse:6.46306
[34]	validation-rmse:6.

2026/07/05 11:29:25 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run luminous-gnu-675 at: http://mlflow:5000/#/experiments/3/runs/c4c9aac6e92045e99a0a2c0c2532fc72
🧪 View experiment at: http://mlflow:5000/#/experiments/3
